# Full-Period Results 2001-2021 (in-sample / descriptive)

Diese Sektion trainiert Random-Forest-Modelle auf dem gesamten Datensatz 2001-2021 und erzeugt deskriptive Metriken, Konfusionsmatrizen, ROC-Kurven und Feature Importances für beide Richtungen. Achtung: Alle Ergebnisse sind in-sample und dürfen nicht als Out-of-Sample-Prognoseleistung interpretiert werden.

Die bestehenden Test-Ergebnisse (Train: 2001-01 bis 2015-12; Test: 2016-01 bis 2021-05) bleiben unverändert und sind weiterhin die maßgebliche Out-of-Sample-Evaluation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve, auc)
import joblib

BASE_DIR = Path(r"d:/Anwendungsprojekt")
PROCESSED_DIR = BASE_DIR / 
 / 

RESULTS_DIR = BASE_DIR / 

PLOTS_DIR = RESULTS_DIR / 

TABLES_DIR = RESULTS_DIR / 

MODELS_DIR = RESULTS_DIR / 

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = PROCESSED_DIR / 

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found at {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
# ensure YearMonth is period-like (string is fine for modeling)
print('Loaded final dataset shape:', df.shape)

In [ ]:
# Derive missing features expected by the RF pipeline (lags and 6-month volatility)
df = df.sort_values([
, 
]).reset_index(drop=True)
# GPR lags: gpr_lag1..3 based on 'gprd_ret' grouped by Index (monthly percent changes)
for lag in (1, 2, 3):
    col = f

    df[col] = df.groupby('Index')['gprd_ret'].shift(lag)
# stock_vol6: 6-month rolling std of 'stock_ret' grouped by Index
if 'stock_vol6' not in df.columns:
    df['stock_vol6'] = df.groupby('Index')['stock_ret'].transform(lambda s: s.rolling(window=6, min_periods=6).std())
# Ensure column naming consistent: lower-case feature names expected below
print('Added gpr_lag1..3 and stock_vol6 where needed. Sample:')
display(df[['YearMonth','Index','gprd_ret','gpr_lag1','gpr_lag2','gpr_lag3','stock_ret','stock_vol6']].head())

In [ ]:
# Feature lists as requested (mapped to existing column names)
features_A = [
    'gprd_ret',      # GPRD_monthly_pct
    'gpr_lag1',
    'gpr_lag2',
    'gpr_lag3',
    'gprd_act_ret',  # GPRD_ACT_monthly_pct
    'gprd_threat_ret', # GPRD_THREAT_monthly_pct
    'GPR_zscore',
    'GPR_spike',
    'Crisis_dummy',
    'Region_encoded',
]
features_B = [
    'stock_ret',    # Stock_monthly_pct
    'stock_ret_lag1',
    'stock_ret_lag2',
    'stock_ret_lag3',
    'stock_vol6',
    'GPR_spike',
    'Crisis_dummy',
    'Region_encoded',
]
# Filter dataset to the final period 2001-01 .. 2021-05 just in case
df_full = df.copy()
# Drop rows with NaNs in the features/targets for a conservative in-sample description
# Direction A target: 'target_stock_down' -> not present in original final_dataset; derive it from 'stock_ret' (<0)
if 'target_stock_down' not in df_full.columns:
    df_full['target_stock_down'] = (df_full['stock_ret'] < 0).astype(int)
# Direction B target already exists: 'target_gpr_up_lead1'
# Drop rows with NaNs in any used columns
needed_cols_A = features_A + ['target_stock_down']
needed_cols_B = features_B + ['target_gpr_up_lead1']
df_A = df_full.dropna(subset=needed_cols_A).copy()
df_B = df_full.dropna(subset=needed_cols_B).copy()
X_A_full = df_A[features_A].copy()
y_A_full = df_A['target_stock_down'].copy()
X_B_full = df_B[features_B].copy()
y_B_full = df_B['target_gpr_up_lead1'].copy()
print('Final shapes:')
print('Direction A X,y:', X_A_full.shape, y_A_full.shape)
print('Direction B X,y:', X_B_full.shape, y_B_full.shape)

In [ ]:
# RF parameters (same as before)
rf_params = dict(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_A_full = RandomForestClassifier(**rf_params)
rf_B_full = RandomForestClassifier(**rf_params)
# Fit on full period (in-sample)
rf_A_full.fit(X_A_full, y_A_full)
rf_B_full.fit(X_B_full, y_B_full)
print('Trained rf_A_full and rf_B_full on full period (in-sample).')

In [ ]:
# Prediction, probabilities and metrics function
def eval_and_save(model, X, y, direction_label, target_name, prefix='direction'):
    y_pred = model.predict(X)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X)[:, 1]
    else:
        y_proba = np.zeros(len(y))
    metrics = dict(
        Sample='Full Period 2001-2021 (in-sample)',
        Direction=direction_label,
        Target=target_name,
        Accuracy=accuracy_score(y, y_pred),
        Precision=precision_score(y, y_pred, zero_division=0),
        Recall=recall_score(y, y_pred, zero_division=0),
        F1=f1_score(y, y_pred, zero_division=0),
        ROC_AUC=roc_auc_score(y, y_proba) if len(np.unique(y))>1 else np.nan,
    )
    # Confusion matrix plot
    cm = confusion_matrix(y, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix {direction_label} {target_name} (full period)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    cm_path = PLOTS_DIR / f'rf_confusion_matrix_{direction_label}_full_period.png'
    plt.savefig(cm_path, bbox_inches='tight')
    plt.close()
    # ROC curve plot
    try:
        fpr, tpr, _ = roc_curve(y, y_proba)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(6,5))
        plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
        plt.plot([0,1],[0,1],'--', color='gray')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve {direction_label} {target_name} (full period)')
        plt.legend(loc='lower right')
        roc_path = PLOTS_DIR / f'rf_roc_curve_{direction_label}_full_period.png'
        plt.savefig(roc_path, bbox_inches='tight')
        plt.close()
    except Exception as exc:
        print('ROC curve could not be computed:', exc)
    # Classification report (text)
    creport = classification_report(y, y_pred, zero_division=0)
    # Feature importances if available
    fi_path = None
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        fi = pd.DataFrame({'feature': X.columns, 'importance': importances})
        fi = fi.sort_values('importance', ascending=False)
        fi_path = TABLES_DIR / f'rf_feature_importance_{direction_label}_full_period.csv'
        fi.to_csv(fi_path, index=False)
        # plot importances
        plt.figure(figsize=(8,6))
        sns.barplot(data=fi, x='importance', y='feature', palette='viridis')
        plt.title(f'Feature Importances {direction_label} (full period)')
        fig_path = PLOTS_DIR / f'rf_feature_importance_{direction_label}_full_period.png'
        plt.tight_layout()
        plt.savefig(fig_path, bbox_inches='tight')
        plt.close()
    # Save model
    model_path = MODELS_DIR / f'random_forest_{direction_label}_full_period.pkl'
    joblib.dump(model, model_path)
    return metrics, creport

In [ ]:
# Evaluate A
metrics_A, report_A = eval_and_save(rf_A_full, X_A_full, y_A_full, 'direction_A', 'target_stock_down')
# Evaluate B
metrics_B, report_B = eval_and_save(rf_B_full, X_B_full, y_B_full, 'direction_B', 'target_gpr_up_lead1')
# Combine metrics and save to CSV
metrics_df = pd.DataFrame([metrics_A, metrics_B])
metrics_out_path = TABLES_DIR / 'rf_metrics_full_period_2001_2021.csv'
metrics_df.to_csv(metrics_out_path, index=False)
print('Saved full-period metrics to', metrics_out_path)
print('
Classification report Direction A:
')
print(report_A)
print('
Classification report Direction B:
')
print(report_B)

Die Full-Period-Ergebnisse beziehen sich auf den gesamten Zeitraum 2001 bis 2021. Da die Modelle auf denselben Daten trainiert und ausgewertet werden, handelt es sich um in-sample-Ergebnisse. Diese Werte dürfen nicht als echte Out-of-Sample-Prognoseleistung interpretiert werden. Für die Bewertung der Prognosefähigkeit bleiben die Test-Ergebnisse 2016 bis 2021 maßgeblich. Die Full-Period-Auswertung dient dazu, die Modellstruktur und Feature Importance über den gesamten Untersuchungszeitraum darzustellen.